In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, RepeatVector, Dense
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("energy consumption.csv")
df["Timestamp"] = pd.to_datetime(df.Timestamp, dayfirst=True)
df = df.sort_values("Timestamp").ffill().fillna(0)
n = int(len(df) * 0.8)
train, test = df[:n], df[n:]
features = df.columns[1:-1]
scaler = StandardScaler().fit(train[train.Anomaly_Label == 0][features])
a = scaler.transform(train[features])
b = scaler.transform(test[features])
labels = train.Anomaly_Label.to_numpy()
x_train = np.array([a[i:i+24] for i in range(len(a)-23) if not labels[i:i+24].any()])
x_test = np.array([b[i:i+24] for i in range(len(b)-23)])

In [ ]:
model = Sequential([Input((24, 5)), LSTM(16), RepeatVector(24),
                    LSTM(16, return_sequences=True), Dense(5)])
model.compile(optimizer="adam", loss="mse")
model.fit(x_train, x_train, epochs=10, batch_size=128)

In [ ]:
error = np.mean((x_train - model.predict(x_train, verbose=0)) ** 2, axis=(1, 2))
threshold = np.percentile(error, 95)
error = np.mean((x_test - model.predict(x_test, verbose=0)) ** 2, axis=(1, 2))
anomalies = error > threshold
print("Anomalies:", anomalies.sum())